# Importing system files

### Make sure you are using the same python version as the env

In [ ]:
import sys, platform
print(sys.executable)
print(platform.python_version())

## Importing necessary packages

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os

# Statsmodels
import statsmodels.api as sm

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error


# Checking the src directory

In [ ]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

# Importing datasets

In [ ]:
BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

In [ ]:
read_from_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ohca_data/ohca_binary_complete.csv")
final_model_df = pd.read_csv(read_from_filepath)
final_model_df = final_model_df.drop(columns=['Unnamed: 0'])

# Checking columns of the dataset

In [ ]:
final_model_df.columns.tolist()

## Creating predictor columns


In [ ]:
predictor_cols = [
       'year', 'month',
       'above_60_proportion', 'male_chinese_proportion',
       'female_chinese_proportion', 'male_malays_proportion',
       'female_malays_proportion', 'male_indians_proportion',
       'female_indians_proportion', 'male_others_proportion',
       'female_others_proportion', 'business_1_encoding',
       'business_2_encoding', 'business_park_encoding', 'school_encoding',
       'airport', 'is_residential'
]

# Keep a reference dataframe for testing/comparison
comparison_df = final_model_df[['pln_area_n', 'subzone_n', 'year', 'month', 'ohca_binary']].copy()
comparison_df

# Implementing Poisson_Distribution_Model

In [ ]:
# Chronological Split
# Training: 2010 - 2019
train_df = final_model_df[final_model_df['year'] <= 2019]
X_train = train_df[predictor_cols].fillna(0)
y_train = train_df['ohca_binary']

# Validation: 2020
val_df = final_model_df[final_model_df['year'] == 2020]
X_val = val_df[predictor_cols].fillna(0)
y_val = val_df['ohca_binary']

# Testing: 2021
test_df = final_model_df[final_model_df['year'] == 2021]
X_test = test_df[predictor_cols].fillna(0)
y_test = test_df['ohca_binary']

In [ ]:
# Add a constant (intercept) to the predictors
X_train_sm = sm.add_constant(X_train)

# Fit the basic Poisson model
poisson_model = sm.GLM(y_train, X_train_sm, family=sm.families.Poisson()).fit()
print(poisson_model.summary())

# Zero-inflated Poisson

In [ ]:
# Define variables that predict whether a zone is an "always zero" state 
# (e.g., non-resi   dential zones, airports, parks)
inflation_cols = ['is_residential', 'airport'] 

# Added in constants column as intercept for inflation model
X_train_zip = sm.add_constant(X_train)
X_infl_zip = sm.add_constant(train_df[inflation_cols])

zip_model = sm.ZeroInflatedPoisson(endog=y_train, 
                                   exog=X_train_zip, 
                                   exog_infl=X_infl_zip, 
                                   inflation='logit').fit()

# Metrics to check

In [ ]:
# Example for test set
test_preds = poisson_model.predict(sm.add_constant(X_test, has_constant='add'))

print(f"RMSE: {np.sqrt(mean_squared_error(y_test, test_preds))}")
print(f"MAE: {mean_absolute_error(y_test, test_preds)}")